<a href="https://colab.research.google.com/github/kvnxmars/llm-regex_mcq-synthesis/blob/main/LLM_Regex_Synthesiser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating the Computational Efficiency, Formal Correctness, and Pedagogical Realism of LLM-Synthesized Regular Expression Assessment Materials

This notebook implements the complete research pipeline for evaluating and comparing **Symbolic Context-Free Grammar (CFG) MCQ generation** against **Probabilistic Neural Generation (using the GPT-5.6 Luna model)**.

### Pipeline Architecture:
1. **Cell 1 & 2: Environment Setup and Dataset Ingestion:** Installs dependencies and parses the ground-truth benchmark questions from Ade-Ibijola's (2018) 10,000 MCQ dataset.
2. **Cell 3: Pedagogical MCQ Wrappers:** Configures the prompt layouts for **Figure 7 (Classical Single-Choice MCQ)** and **Figure 8 (Multi-Answer MCQ)**.
3. **Cell 4: Neural Synthesis & Custom Cost Logging:** Orchestrates parallel LLM generation runs using GPT-5.6 Luna, implementing token-level cost tracking (including caching) and latency metrics.
Cell 1 (Code)

In [1]:
#environment setup & libraries
print("Installing libraries..")

#install opneai for gpt4o prompting and fpdf2 for pdf generation
!pip install --quiet openai fpdf2 pypdf pdfplumber
print("Libraries successfully installed!")

print("Assembling imports..")
import os
import re
import sys
import time
import random
import pypdf
import json

from google.colab import files

print("Setup complete.. Ready to load pdf.")

Installing libraries..
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.0/337.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 26.7 MB/s eta 0:00:00
Libraries successfully installed!
Assembling imports..
Setup complete.. Ready to load pdf.


## Cell 2: Ground-Truth Dataset Parsing and Text Normalization


This cell ingests both uploaded baseline PDFs and merges them into a single, unified database (`baseline_tasks.json`) of exactly **500 benchmark tasks**.

The current pipeline extracts:

A. **Single-Answer Baseline:** Parses the first **250 tasks** from `10000 iterations of MCQs and Solutions.pdf` (Figure 7 layout).

B. **Multi-Answer Baseline:** Parses the first **250 tasks** from `1000 iterations of MCQs - multiple correct solutions.pdf` (Figure 8 layout).

### Normalization and Cleansing:
* **Asterisk Normalization:** Converts mathematical Unicode stars (\\(\ast\\) or `\u2217`) to standard keyboard stars (`*`) to ensure compatibility with our automata engine.
* **Alphabet Boundary Restricting:** Dynamically extracts the restricted alphabet (\\(\Sigma\\)) for each question, filtering out spaces and empty-string lambda symbols (\\(\lambda\\)).
* **Deferred Multi-Answer Keys:** For multi-answer tasks, `labeled_correct` is initialized to `null`. Cell 5 will programmatically grade and label these options using our local DFA verifier.






In [2]:
# ============================================================
# CELL 2A: PARSE 10,000-ITEM SINGLE-ANSWER PDF (Figure 7)
# ============================================================
import os, re, json
import pdfplumber
from google.colab import drive, files

if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')
OUTPUT_DIR   = "/content/drive/MyDrive/regex_llm_project"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SINGLE_PDF = "10000 iterations of MCQs and Solutions.pdf"
if not os.path.exists(SINGLE_PDF):
  print(f"Please upload '{SINGLE_PDF}':")
  files.upload()

def clean_regex_text(raw):
  return re.sub(r"\s+", "", raw).replace("*", "*")

print("Extracting text (column-aware, ~5s for 36 pages)..")
text_parts = []
with pdfplumber.open(SINGLE_PDF) as pdf:
    for page in pdf.pages:
      w, h = page.width, page.height
      text_parts.append(page.crop((0, 0, w / 2, h)).extract_text() or "")
      text_parts.append(page.crop((w / 2, 0, w, h)).extract_text() or "")
text = "\n".join(text_parts)

# A wildcard can never consume the start of the next "(N)." item -- a malformed
# item fails cleanly instead of silently swallowing its neighbor's fields.
NB = r"(?:(?!\(\d+\)\.).)*?"
pattern = re.compile(
    r"\((\d+)\)\.\s*Rex\s*=\s*(" + NB + r");\s*Σ\s*=\s*\{([^}\n]+)\}\.\s*"
    r"(" + NB + r")Answer:\s*([A-E])\.",
    re.DOTALL
)
opt_pattern = re.compile(r"([A-E])\.\s*(.*?)\s*(?=(?:[A-E]\.)|$)", re.DOTALL)

all_matches = list(pattern.finditer(text))
matched_ids = {int(m.group(1)) for m in all_matches}
skipped = sorted(set(range(1, 10001)) - matched_ids)

COUNT = 250
tasks = []
for m in all_matches:
    task_id = int(m.group(1))
    target_regex = clean_regex_text(m.group(2))
    alphabet = [s.strip() for s in m.group(3).replace(" ", "").split(",") if s.strip() and s.strip() != "λ"]
    answer_letter = m.group(5)
    option_block = m.group(4)
    letter_to_text = {letter: opt_text.strip() for letter, opt_text in opt_pattern.findall(option_block)}
    options = [{"string": letter_to_text.get(l, ""), "labeled_correct": (l == answer_letter)} for l in "ABCDE"]

    tasks.append({
        "task_id": task_id,
        "wrapper_type": "Figure 7 (Single-Choice)",
        "target_regex": target_regex,
        "alphabet": alphabet,
        "options": options
    })
    if len(tasks) >= COUNT:
        break

out_path = os.path.join(OUTPUT_DIR, "single_tasks.json")
with open(out_path, "w") as f:
    json.dump(tasks, f, indent=2)

# ---- SNIPPET ONLY ----
print(f"\n✅ Matched {len(all_matches)}/10000 items ({len(skipped)} malformed & excluded: {skipped[:10]}{'...' if len(skipped) > 10 else ''})")
print(f"💾 Saved first {len(tasks)} tasks to '{out_path}'\n")
print("--- Spot check (compare directly against the PDF page) ---")
for t in tasks[:3]:
    correct = next(o["string"] for o in t["options"] if o["labeled_correct"])
    print(f"  #{t['task_id']}: r = {t['target_regex']}  Σ={t['alphabet']}  →  correct: \"{correct}\"")


Mounted at /content/drive
Please upload '10000 iterations of MCQs and Solutions.pdf':


Saving 10000 iterations of MCQs and Solutions.pdf to 10000 iterations of MCQs and Solutions.pdf
Extracting text (column-aware, ~5s for 36 pages)..

✅ Matched 10000/10000 items (0 malformed & excluded: [])
💾 Saved first 250 tasks to '/content/drive/MyDrive/regex_llm_project/single_tasks.json'

--- Spot check (compare directly against the PDF page) ---
  #1: r = f+j∗  Σ=['l', 'f', 'j']  →  correct: "ffff"
  #2: r = b∗f?  Σ=['z', 'r', 'b', 'f']  →  correct: "bbb"
  #3: r = q+i?  Σ=['y', 'n', 'q', 'i']  →  correct: "qi"


In [3]:
# ============================================================
# CELL 2B: PARSE 1,000-ITEM MULTI-ANSWER PDF (Figure 8) + MERGE
# ============================================================
import os, re, json
import pdfplumber
from google.colab import files

MULTI_PDF = "1000 iterations of MCQs - multiple correct solutions.pdf"
if not os.path.exists(MULTI_PDF):
    print(f"Please upload '{MULTI_PDF}':")
    files.upload()

def clean_regex_text(raw):
    return re.sub(r"\s+", "", raw).replace("∗", "*")

print("🔄 Extracting text (column-aware, ~5s for 36 pages)...")
text_parts = []
with pdfplumber.open(MULTI_PDF) as pdf:
    for page in pdf.pages:
        w, h = page.width, page.height
        text_parts.append(page.crop((0, 0, w / 2, h)).extract_text() or "")
        text_parts.append(page.crop((w / 2, 0, w, h)).extract_text() or "")
text = "\n".join(text_parts)

NB = r"(?:(?!\(No\.\d+\)\.).)*?"
pattern = re.compile(
    r"\(No\.\s*(\d+)\)\.\s*Let\s*r\s*=\s*(" + NB + r")\.\s*"
    r"Which\s*of\s*the\s*following\s*strings\s*is\s*in\s*L\s*\(r\)\?\s*Take\s*"
    r"Σ\s*=\s*\{([^}\n]+)\}\.\s*"
    r"(" + NB + r")Circle\s*your\s*answers\.",
    re.DOTALL
)

all_matches = list(pattern.finditer(text))
matched_ids = {int(m.group(1)) for m in all_matches}
skipped = sorted(set(range(1, 1001)) - matched_ids)

# These answers are UNDERLINED in the PDF, not text-labeled. pdfplumber's text
# extraction can't see underline formatting, so labeled_correct stays None here
# and is resolved programmatically by the DFA engine in Cell 5.
COUNT = 250
tasks = []
for m in all_matches:
    task_id = int(m.group(1)) + 10000   # offset, avoids colliding with single-answer IDs
    target_regex = clean_regex_text(m.group(2))
    alphabet = [s.strip() for s in m.group(3).replace(" ", "").split(",") if s.strip() and s.strip() != "λ"]
    opts = [o.strip() for o in m.group(4).strip().rstrip(",").split(",") if o.strip()]
    options = [{"string": o, "labeled_correct": None} for o in opts]

    tasks.append({
        "task_id": task_id,
        "wrapper_type": "Figure 8 (Multi-Answer)",
        "target_regex": target_regex,
        "alphabet": alphabet,
        "options": options
    })
    if len(tasks) >= COUNT:
        break

OUTPUT_DIR = "/content/drive/MyDrive/regex_llm_project"
out_path = os.path.join(OUTPUT_DIR, "multi_tasks.json")
with open(out_path, "w") as f:
    json.dump(tasks, f, indent=2)

print(f"\n✅ Matched {len(all_matches)}/1000 items ({len(skipped)} malformed & excluded: {skipped})")
print(f"💾 Saved first {len(tasks)} tasks to '{out_path}'\n")
print("--- Spot check (labeled_correct is null by design) ---")
for t in tasks[:3]:
    print(f"  #{t['task_id']}: r = {t['target_regex']}  Σ={t['alphabet']}  options={[o['string'] for o in t['options']]}")

# ---- MERGE with Cell 2A's output into the unified baseline ----
with open(os.path.join(OUTPUT_DIR, "single_tasks.json")) as f:
    single_tasks = json.load(f)

unified = single_tasks + tasks
baseline_path = os.path.join(OUTPUT_DIR, "baseline_tasks.json")
with open(baseline_path, "w") as f:
    json.dump(unified, f, indent=2)

print(f"\n💾 Merged {len(single_tasks)} single + {len(tasks)} multi = {len(unified)} → '{baseline_path}'")

Please upload '1000 iterations of MCQs - multiple correct solutions.pdf':


Saving 1000 iterations of MCQs - multiple correct solutions.pdf to 1000 iterations of MCQs - multiple correct solutions.pdf
🔄 Extracting text (column-aware, ~5s for 36 pages)...

✅ Matched 992/1000 items (8 malformed & excluded: [4, 49, 82, 96, 171, 185, 678, 692])
💾 Saved first 250 tasks to '/content/drive/MyDrive/regex_llm_project/multi_tasks.json'

--- Spot check (labeled_correct is null by design) ---
  #10001: r = g*yi+  Σ=['e', 't', 'w', 'g', 'y', 'i']  options=['ettttii', 'twwwgggyyiiii', 'eetttwwggi', 'ttwwwyyi', 'gyiiii']
  #10002: r = l*f?  Σ=['h', 'y', 'l', 'f']  options=['yy', 'h', 'llllf', 'l', 'yyyyf']
  #10003: r = l+vd*  Σ=['n', 'y', 'p', 'l', 'v', 'd']  options=['nnppp', 'llvd', 'yyllvvd', 'llllv', 'nnyyypppllllvvdd']

💾 Merged 250 single + 250 multi = 500 → '/content/drive/MyDrive/regex_llm_project/baseline_tasks.json'


## Cell 3: Pedagogical MCQ Wrappers &amp; Strict Operator Constraints

This cell configures the **System Prompt** and **User Prompt Templates** for Figure 7 and Figure 8 MCQ wrappers.
### FLA Operator Constraints:
To isolate the neural model's logical reasoning engine rather than its memory of programming-specific regex shorthand (such as \`\\d\`, \`\\w\`, or anchors like \`^\` and \`$\`), the system prompt strictly constrains the LLM to use only core theoretical operators:

\* Concatenation (implied)

\* Alternation / Union (\`|\`)

\* Kleene Star (\`\*\`)

\* Kleene Plus (\`+\`)

\* Optionality (\`?\`)

\* Explicit grouping parentheses (\`()\`)

In [4]:
# =====================================================================
# CELL 3: MCQ WRAPPERS & SYSTEM PROMPTS (FIGURE 7 & FIGURE 8)
# =====================================================================

SYSTEM_PROMPT = """
You are an expert tutor in Formal Languages and Automata (FLA) theory. Your task is to generate a pedagogically sound Multiple-Choice Question (MCQ) item for an undergraduate Computer Science practice exam.

You must strictly adhere to the following language and syntax constraints:
1. ALPHABET CONSTRAINT (Σ): You may ONLY use the characters explicitly provided in the user's alphabet. No other characters or shorthand symbols (such as \\d, \\w, \\s) are allowed.
2. OPERATOR CONSTRAINT: You may ONLY use the core theoretical operators of regular expressions:
   - Concatenation (implied, e.g., ab)
   - Alternation / Union (using "|" or "+", e.g., a|b)
   - Kleene Star ("*")
   - Kleene Plus ("+")
   - Optionality ("?")
   - Explicit grouping parentheses ("( )")
   Do NOT use lookarounds, backreferences, anchors (^ or $), or any PCRE programming extensions.

You must respond with STRICT JSON ONLY. Do not wrap the JSON in markdown code fences, and do not include any conversational preamble or postscript.

JSON SCHEMA:
{
  "synthesized_regex": "<the regular expression r_LLM you constructed>",
  "shuffled_options": [
    {"label": "A", "string": "<string_1>", "labeled_correct": true},
    {"label": "B", "string": "<string_2>", "labeled_correct": false},
    {"label": "C", "string": "<string_3>", "labeled_correct": false},
    {"label": "D", "string": "<string_4>", "labeled_correct": false},
    {"label": "E", "string": "<string_5>", "labeled_correct": false}
  ],
  "pedagogical_rationales": {
    "regex_design": "Explain in 1 sentence how you constructed the regex.",
    "option_mapping": {
       "A": "Why this string matches/does not match.",
       "B": "Why this string matches/does not match.",
       "C": "Why this string matches/does not match.",
       "D": "Why this string matches/does not match.",
       "E": "Why this string matches/does not match."
    }
  }
}
"""

# Wrapper 1: Figure 7 Classic Format
WRAPPER_FIG7_TEMPLATE = """
Target Language Specification: {description}
Alphabet (Σ): {{{alphabet_symbols}}}

Generate one classical MCQ item following this exact problem wrapper layout from Figure 7:

"For each of the following regular expressions, defined over the indicated alphabet, determine (from options A to E) which string is accepted by the expression.
Let r = {target_regex}. Take Σ = {{{alphabet_symbols}}}.

[List five randomly shuffled strings here; four wrong and one correct answer.]"

Instructions:
1. Synthesize the regular expression (it must be equivalent to {target_regex}).
2. Generate exactly FIVE options (labeled A to E).
3. Exactly ONE option must be mathematically correct (labeled_correct = true), and the other FOUR must be incorrect distractors.

Provide your output strictly matching the required JSON schema.
"""

# Wrapper 2: Figure 8 Multi-Answer Format
WRAPPER_FIG8_TEMPLATE = """
Target Language Specification: {description}
Alphabet (Σ): {{{alphabet_symbols}}}
Number of Correct Options (m): {num_correct}

Generate one multi-answer MCQ item following this exact problem wrapper layout from Figure 8:

"Let r = [synthesized_regex]. Which of the following strings is in L(r)? Take Σ = {{{alphabet_symbols}}}.
[List 5 randomly shuffled strings here; {num_correct} correct strings, and {num_wrong} wrong strings.]"

Instructions:
1. Synthesize the regular expression (it must be equivalent to {target_regex}).
2. Generate exactly FIVE options (labeled A to E).
3. Exactly {num_correct} options must be mathematically correct (labeled_correct = true), and the remaining {num_wrong} options must be incorrect distractors (where {num_correct} must be between 1 and 4).

Provide your output strictly matching the required JSON schema.
"""

print("✅ Step 3 complete! Prompt templates for Figure 7 and Figure 8 are compiled.")

✅ Step 3 complete! Prompt templates for Figure 7 and Figure 8 are compiled.




## Cell 4A: Free Llama-3.1 Synthesis Pipeline (Groq Engine)

This cell executes the 500-task benchmark using **Meta's Llama-3.1-70B** hosted on Groq's high-speed LPU infrastructure.

### Features:
- **Zero-Cost Execution:** Runs entirely on Groq's free developer tier (API key from console.groq.com).
- **High Throughput:** Executes at ~250 tokens/second without artificial daily rate limit caps.
- **Dedicated Output:** Saves results to `cumulative_tasks_llama.json` to preserve data for comparative model analysis against OpenAI.






In [5]:
# =====================================================================
# CELL 4A: FREE GROQ (LLAMA / GPT-OSS) TEST-SET / SYNTHESIS LOOP
# =====================================================================
import os, sys, time, json, random
from openai import OpenAI
from getpass import getpass

# -- 0. Mount Drive so incremental progress survives a disconnect --
# Colab's local disk (/content/...) is fine WITHIN a session — that's
# how your earlier runs saved cumulative_tasks_llama.json successfully.
# But if the runtime disconnects (idle timeout, ~90min job), that disk
# is wiped. Mount Drive once and write there instead.
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = "/content/drive/MyDrive/regex_llm_project"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -- 1. Groq Provider Configuration --
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROQ_MODEL = "openai/gpt-oss-120b"   # <-- STRING, not a list

GROQ_API_KEY = getpass("Enter Groq API key (console.groq.com): ")
groq_client = OpenAI(base_url=GROQ_BASE_URL, api_key=GROQ_API_KEY)

# -- 2. TEST-SET LIMIT --
# None = full 500-task run. Set to 5 or 100 to eyeball output before committing.
MAX_TASKS = 5

# -- 3. Ingest baseline dataset --
baseline_path = os.path.join(OUTPUT_DIR, "baseline_tasks.json")
if not os.path.exists(baseline_path) and os.path.exists("baseline_tasks.json"):
    baseline_path = "baseline_tasks.json"   # fallback if Cell 2 hasn't been pointed at Drive yet

try:
    with open(baseline_path, "r") as f:
        all_baseline_tasks = json.load(f)
    print(f"Loaded {len(all_baseline_tasks)} benchmark tasks from '{baseline_path}'.")
except FileNotFoundError:
    print("Error! baseline_tasks.json missing. Run Cell 2 first.")
    sys.exit(1)

# -- 4. Resume log: ONLY successful tasks are ever written here --
llama_output_file = os.path.join(OUTPUT_DIR, "cumulative_tasks_llama.json")
llama_dataset = []
if os.path.exists(llama_output_file):
    with open(llama_output_file, "r") as f:
        llama_dataset = json.load(f)
    print(f"Resuming: {len(llama_dataset)} tasks already completed.")
existing_ids = {item["task_id"] for item in llama_dataset}

# -- 5. Execution loop --
print(f"\nLaunching {GROQ_MODEL} (Groq) Synthesis Loop..")
print("----------------------------------------------")

processed_this_run = 0
for idx, task in enumerate(all_baseline_tasks):
    if MAX_TASKS is not None and processed_this_run >= MAX_TASKS:
        print(f"\nHit MAX_TASKS={MAX_TASKS}. Stopping for review.")
        break

    task_id = task["task_id"]
    if task_id in existing_ids:
        continue

    target_regex = task["target_regex"]
    alphabet_list = task["alphabet"]
    alphabet_str = ", ".join(alphabet_list)
    description = f"strings matching the regular expression {target_regex}"
    wrapper_type = task["wrapper_type"]

    if wrapper_type == "Figure 7 (Single-Choice)":
        user_prompt = WRAPPER_FIG7_TEMPLATE.format(
            description=description, target_regex=target_regex, alphabet_symbols=alphabet_str
        )
    else:
        rng = random.Random(task_id)          # seeded -> identical prompt every run/model
        num_correct = rng.randint(2, 4)
        user_prompt = WRAPPER_FIG8_TEMPLATE.format(
            description=description, target_regex=target_regex, alphabet_symbols=alphabet_str,
            num_correct=num_correct, num_wrong=5 - num_correct
        )

    print(f"[{GROQ_MODEL}] Task {idx + 1}/{len(all_baseline_tasks)} [ID: {task_id}] | {wrapper_type}...")

    parsed_response = None
    prompt_tokens = completion_tokens = 0
    parse_error = None

    for attempt in range(4):
        t0 = time.perf_counter()
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.1,
                response_format={"type": "json_object"}
            )
            duration = time.perf_counter() - t0
            prompt_tokens = response.usage.prompt_tokens
            completion_tokens = response.usage.completion_tokens
            raw_content = response.choices[0].message.content.strip()   # <-- [0] added
            parsed_response = json.loads(raw_content)
            parse_error = None
            break
        except Exception as e:
            duration = time.perf_counter() - t0
            parse_error = str(e)
            if attempt < 3:
                wait = 2 * (2 ** attempt)
                print(f"  attempt {attempt+1} failed ({parse_error[:80]}); retry in {wait}s")
                time.sleep(wait)

    if parsed_response is None:
        print(f"  GIVING UP on task {task_id}: {parse_error}")
        continue   # NOT added to existing_ids -> will retry next run

    record = {
        "task_id": task_id,
        "model_engine": GROQ_MODEL,
        "wrapper_type": wrapper_type,
        "target_regex": target_regex,
        "alphabet": alphabet_list,
        "synthesized_regex": parsed_response.get("synthesized_regex"),
        "options": parsed_response.get("shuffled_options"),
        "rationales": parsed_response.get("pedagogical_rationales"),
        "metrics": {
            "duration_s": duration,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "parse_error": None,
        }
    }

    llama_dataset.append(record)
    existing_ids.add(task_id)
    processed_this_run += 1

    with open(llama_output_file, "w") as f:
        json.dump(llama_dataset, f, indent=4)

    print(f"  {duration:.2f}s | in: {prompt_tokens} | out: {completion_tokens}")
    time.sleep(0.5)
    print("-------------------------------------------------------------")

print(f"\nRun complete! {len(llama_dataset)} tasks saved to '{llama_output_file}'.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Enter Groq API key (console.groq.com): ··········
Loaded 500 benchmark tasks from '/content/drive/MyDrive/regex_llm_project/baseline_tasks.json'.
Resuming: 5 tasks already completed.

Launching openai/gpt-oss-120b (Groq) Synthesis Loop..
----------------------------------------------
[openai/gpt-oss-120b] Task 6/500 [ID: 6] | Figure 7 (Single-Choice)...
  6.04s | in: 769 | out: 1145
-------------------------------------------------------------
[openai/gpt-oss-120b] Task 7/500 [ID: 7] | Figure 7 (Single-Choice)...
  2.20s | in: 770 | out: 933
-------------------------------------------------------------
[openai/gpt-oss-120b] Task 8/500 [ID: 8] | Figure 7 (Single-Choice)...
  2.33s | in: 772 | out: 953
-------------------------------------------------------------
[openai/gpt-oss-120b] Task 9/500 [ID: 9] | Figure 7 (Single-Choice)...
  2.82s | in: 778 | out: 120

This cell is to test the results of the test set experiment (take a peek into the dataset).

In [6]:
import json
with open(os.path.join(OUTPUT_DIR, "cumulative_tasks_llama.json")) as f:
    print(json.dumps(json.load(f)[:2], indent=2))

[
  {
    "task_id": 1,
    "model_engine": "openai/gpt-oss-120b",
    "wrapper_type": "Figure 7 (Single-Choice)",
    "target_regex": "f+j\u2217",
    "alphabet": [
      "l",
      "f",
      "j"
    ],
    "synthesized_regex": "(f|j*)",
    "options": [
      {
        "label": "A",
        "string": "l",
        "labeled_correct": false
      },
      {
        "label": "B",
        "string": "jjj",
        "labeled_correct": true
      },
      {
        "label": "C",
        "string": "fl",
        "labeled_correct": false
      },
      {
        "label": "D",
        "string": "ff",
        "labeled_correct": false
      },
      {
        "label": "E",
        "string": "jf",
        "labeled_correct": false
      }
    ],
    "rationales": {
      "regex_design": "Combined the single\u2011character alternative f with j* to capture either f or any number of j's (including none).",
      "option_mapping": {
        "A": "Contains the symbol l, which is not allowed by the regex,

## Cell 5: Pure-Python Formal Verification Engine (Thompson NFA → Subset DFA &amp; Option Judge)
This cell implements our formal verification judge under the **∀uto∃val framework** (Karia et al., 2024). It operates purely in Python without external C-libraries or system dependencies.
### Verification Capabilities:
1. **Thompson's Construction (RE → NFA):** Parses FLA operators (`concatenation`, `|`, `*`, `+`, `?`, `()`) into Non-deterministic Finite Automata with \\(\epsilon\\)-transitions.
2. **Subset Construction (NFA → DFA):** Converts NFAs into Deterministic Finite Automata over the problem's restricted alphabet (\\(\\Sigma\\)).
3. **Formal Equivalence Check (\\(r\_{\text{target}} \equiv r_{\text{LLM}}\\)):** Tests if the LLM's synthesized regex accepts the exact same language as the benchmark target regex across bounded string generations.
4. **Option Soundness &amp; Distractor Leak Detection:** - Verifies if options marked as correct by the LLM are accepted by the target DFA. - Flags any distractor accepted by the target DFA as a **Distractor Leak** (a broken MCQ with multiple valid answers).
5. **Consistency Rate Calculation:** Computes the overall formal correctness percentage for the run.

In [7]:
# =====================================================================
# CELL 5A: PURE-PYTHON REGEX → NFA → DFA ENGINE
# =====================================================================

class State:
    def __init__(self, is_end=False):
        self.is_end = is_end
        self.transitions = {}   # symbol -> list[State]
        self.epsilon = []       # epsilon transitions


class NFA:
    def __init__(self, start, end):
        self.start = start
        self.end = end


class DFA:
    def __init__(self, start_state, transitions, accept_states, alphabet):
        self.start_state = start_state
        self.transitions = transitions
        self.accept_states = accept_states
        self.alphabet = set(alphabet)

    def accepts(self, input_str):
        current = self.start_state

        for symbol in input_str:
            if symbol not in self.alphabet:
                return False

            if current not in self.transitions:
                return False

            if symbol not in self.transitions[current]:
                return False

            current = self.transitions[current][symbol]

        return current in self.accept_states


def re_to_postfix(regex):
    """
    Convert a restricted regular expression to postfix notation.

    Supported operators:
        |   alternation
        .   explicit concatenation
        *   zero or more
        +   one or more
        ?   zero or one
        ()  grouping
    """

    if not isinstance(regex, str):
        raise TypeError("Regex must be a string.")

    regex = regex.replace(" ", "").replace("∗", "*")

    if not regex:
        raise ValueError("Regex cannot be empty.")

    precedence = {
        "|": 1,
        ".": 2,
    }

    # Insert explicit concatenation operators.
    formatted = []

    for i, c1 in enumerate(regex):
        formatted.append(c1)

        if i + 1 >= len(regex):
            continue

        c2 = regex[i + 1]

        left_can_concat = (
            c1 not in "(|"
        )

        right_can_concat = (
            c2 not in ")|*+?"
        )

        if left_can_concat and right_can_concat:
            formatted.append(".")

    output = []
    stack = []

    operators = {"|", ".", "*", "+", "?", "(", ")"}

    for char in formatted:

        if char not in operators:
            output.append(char)

        elif char == "(":
            stack.append(char)

        elif char == ")":
            found_open = False

            while stack:
                top = stack.pop()

                if top == "(":
                    found_open = True
                    break

                output.append(top)

            if not found_open:
                raise ValueError("Unmatched ')' in regex.")

        elif char in {"*", "+", "?"}:
            # Unary operators apply directly to the previous expression.
            output.append(char)

        else:
            while (
                stack
                and stack[-1] != "("
                and precedence[stack[-1]] >= precedence[char]
            ):
                output.append(stack.pop())

            stack.append(char)

    while stack:
        top = stack.pop()

        if top == "(":
            raise ValueError("Unmatched '(' in regex.")

        output.append(top)

    return "".join(output)


def postfix_to_nfa(postfix):
    """Thompson construction: postfix regex → NFA."""

    if not postfix:
        raise ValueError("Cannot construct NFA from empty postfix expression.")

    stack = []

    for char in postfix:

        # --------------------------------------------------------------
        # Literal symbol
        # --------------------------------------------------------------
        if char not in {"*", "+", "?", "|", "."}:
            start = State()
            end = State(is_end=True)

            start.transitions[char] = [end]

            stack.append(NFA(start, end))

        # --------------------------------------------------------------
        # Kleene star
        # --------------------------------------------------------------
        elif char == "*":
            if not stack:
                raise ValueError("'*' has no operand.")

            nfa = stack.pop()

            start = State()
            end = State(is_end=True)

            start.epsilon.extend([nfa.start, end])

            nfa.end.is_end = False
            nfa.end.epsilon.extend([nfa.start, end])

            stack.append(NFA(start, end))

        # --------------------------------------------------------------
        # One or more
        # --------------------------------------------------------------
        elif char == "+":
            if not stack:
                raise ValueError("'+' has no operand.")

            nfa = stack.pop()

            start = State()
            end = State(is_end=True)

            start.epsilon.append(nfa.start)

            nfa.end.is_end = False
            nfa.end.epsilon.extend([nfa.start, end])

            stack.append(NFA(start, end))

        # --------------------------------------------------------------
        # Optional
        # --------------------------------------------------------------
        elif char == "?":
            if not stack:
                raise ValueError("'?' has no operand.")

            nfa = stack.pop()

            start = State()
            end = State(is_end=True)

            start.epsilon.extend([nfa.start, end])

            nfa.end.is_end = False
            nfa.end.epsilon.append(end)

            stack.append(NFA(start, end))

        # --------------------------------------------------------------
        # Alternation
        # --------------------------------------------------------------
        elif char == "|":
            if len(stack) < 2:
                raise ValueError("'|' requires two operands.")

            right = stack.pop()
            left = stack.pop()

            start = State()
            end = State(is_end=True)

            start.epsilon.extend([
                left.start,
                right.start
            ])

            left.end.is_end = False
            right.end.is_end = False

            left.end.epsilon.append(end)
            right.end.epsilon.append(end)

            stack.append(NFA(start, end))

        # --------------------------------------------------------------
        # Concatenation
        # --------------------------------------------------------------
        elif char == ".":
            if len(stack) < 2:
                raise ValueError("'.' requires two operands.")

            right = stack.pop()
            left = stack.pop()

            left.end.is_end = False
            left.end.epsilon.append(right.start)

            stack.append(NFA(left.start, right.end))

    if len(stack) != 1:
        raise ValueError(
            f"Malformed regex: expected one NFA, got {len(stack)}."
        )

    return stack[0]


def get_epsilon_closure(states):
    """Return the epsilon closure of a set of NFA states."""

    closure = set(states)
    stack = list(states)

    while stack:
        state = stack.pop()

        for next_state in state.epsilon:
            if next_state not in closure:
                closure.add(next_state)
                stack.append(next_state)

    return closure


def nfa_to_dfa(nfa, alphabet):
    """Subset construction: NFA → DFA."""

    if nfa is None:
        raise ValueError("NFA cannot be None.")

    alphabet = sorted(set(alphabet))

    initial_closure = frozenset(
        get_epsilon_closure([nfa.start])
    )

    dfa_states = {
        initial_closure: 0
    }

    dfa_transitions = {}
    accept_states = set()

    if any(state.is_end for state in initial_closure):
        accept_states.add(0)

    unmarked = [initial_closure]
    next_state_id = 1

    while unmarked:

        current_set = unmarked.pop(0)
        current_id = dfa_states[current_set]

        dfa_transitions[current_id] = {}

        for symbol in alphabet:

            next_states = set()

            for nfa_state in current_set:

                if symbol in nfa_state.transitions:
                    next_states.update(
                        nfa_state.transitions[symbol]
                    )

            if not next_states:
                continue

            next_closure = frozenset(
                get_epsilon_closure(next_states)
            )

            if next_closure not in dfa_states:

                dfa_states[next_closure] = next_state_id

                if any(
                    state.is_end
                    for state in next_closure
                ):
                    accept_states.add(next_state_id)

                next_state_id += 1
                unmarked.append(next_closure)

            dfa_transitions[current_id][symbol] = (
                dfa_states[next_closure]
            )

    return DFA(
        start_state=0,
        transitions=dfa_transitions,
        accept_states=accept_states,
        alphabet=alphabet
    )


def compile_regex_to_dfa(regex, alphabet):
    """Compile a restricted regex directly into a DFA."""

    postfix = re_to_postfix(regex)
    nfa = postfix_to_nfa(postfix)

    return nfa_to_dfa(
        nfa,
        alphabet
    )

In [8]:
# =====================================================================
# CELL 5B: EXACT DFA LANGUAGE EQUIVALENCE
# =====================================================================

from collections import deque


def check_dfa_equivalence(dfa1, dfa2):
    """
    Determine whether two DFAs recognize exactly the same language.

    Uses BFS over the product automaton.

    Returns:
        (True, None)
        or
        (False, counterexample)
    """

    alphabet = sorted(
        set(dfa1.alphabet) | set(dfa2.alphabet)
    )

    # Each product state is:
    #     (state in DFA1, state in DFA2, input string)
    queue = deque([
        (
            dfa1.start_state,
            dfa2.start_state,
            ""
        )
    ])

    visited = {
        (
            dfa1.start_state,
            dfa2.start_state
        )
    }

    while queue:

        state1, state2, witness = queue.popleft()

        accepted1 = state1 in dfa1.accept_states
        accepted2 = state2 in dfa2.accept_states

        # If one accepts and the other rejects,
        # the witness proves inequivalence.
        if accepted1 != accepted2:
            return False, witness

        for symbol in alphabet:

            next1 = dfa1.transitions.get(
                state1, {}
            ).get(symbol)

            next2 = dfa2.transitions.get(
                state2, {}
            ).get(symbol)

            # Missing transition = implicit rejecting sink.
            key = (next1, next2)

            if key not in visited:
                visited.add(key)

                queue.append(
                    (
                        next1,
                        next2,
                        witness + symbol
                    )
                )

    return True, None

In [9]:
# =====================================================================
# CELL 5C: DATASET FORMAL VERIFICATION AUDITOR
# =====================================================================

import json
import os
import copy


INPUT_FILE = "cumulative_tasks_llama.json"
OUTPUT_FILE = "audited_tasks_llama.json"


def audit_task(task):
    """Formally audit one generated task."""

    task_id = task.get("task_id")

    target_regex = task.get("target_regex")
    synth_regex = task.get("synthesized_regex")

    alphabet = task.get("alphabet", [])
    options = task.get("options", [])

    result = copy.deepcopy(task)

    verification = {
        "target_compilation": False,
        "synthesis_compilation": None,
        "equivalence_passed": None,
        "counterexample": None,
        "options_audited": bool(options),
        "options_sound": None,
        "distractor_leak_detected": False,
        "fully_consistent": False,
        "errors": []
    }

    # --------------------------------------------------------------
    # 1. Compile target
    # --------------------------------------------------------------

    try:

        target_dfa = compile_regex_to_dfa(
            target_regex,
            alphabet
        )

        verification["target_compilation"] = True

    except Exception as e:

        verification["errors"].append(
            f"Target regex compilation failed: {e}"
        )

        result["verification"] = verification

        return result


    # --------------------------------------------------------------
    # 2. Compile + compare synthesized regex
    # --------------------------------------------------------------

    if synth_regex is not None:

        try:

            synth_dfa = compile_regex_to_dfa(
                synth_regex,
                alphabet
            )

            verification["synthesis_compilation"] = True

            equivalent, counterexample = check_dfa_equivalence(
                target_dfa,
                synth_dfa
            )

            verification["equivalence_passed"] = equivalent
            verification["counterexample"] = counterexample

        except Exception as e:

            verification["synthesis_compilation"] = False

            verification["errors"].append(
                f"Synthesized regex compilation failed: {e}"
            )

    else:

        verification["synthesis_compilation"] = False

        verification["errors"].append(
            "No synthesized regex provided."
        )


    # --------------------------------------------------------------
    # 3. Audit MCQ options
    # --------------------------------------------------------------

    if options:

        options_sound = True
        distractor_leak = False

        for option in options:

            option_string = (
                option.get("string", "")
                .strip()
            )

            labelled_correct = bool(
                option.get(
                    "labeled_correct",
                    False
                )
            )

            actual_accepts = target_dfa.accepts(
                option_string
            )

            option["actual_target_match"] = (
                actual_accepts
            )

            # Correct label but invalid answer
            if labelled_correct and not actual_accepts:

                options_sound = False

                option["verification_issue"] = (
                    "Labeled correct but rejected by target DFA."
                )

            # Distractor actually belongs to target language
            elif (
                not labelled_correct
                and actual_accepts
            ):

                distractor_leak = True

                option["verification_issue"] = (
                    "Distractor leak: rejected label but "
                    "accepted by target DFA."
                )

        verification["options_sound"] = options_sound
        verification["distractor_leak_detected"] = (
            distractor_leak
        )

    # --------------------------------------------------------------
    # 4. Overall consistency
    # --------------------------------------------------------------

    verification["fully_consistent"] = (
        verification["target_compilation"]
        and verification["synthesis_compilation"] is True
        and verification["equivalence_passed"] is True
        and (
            not options
            or (
                verification["options_sound"] is True
                and not verification["distractor_leak_detected"]
            )
        )
    )

    result["verification"] = verification

    return result

In [10]:
def run_audit(input_file, output_file):

    if not os.path.exists(input_file):

        raise FileNotFoundError(
            f"Input file '{input_file}' not found. "
            "Run the synthesis cell first."
        )

    with open(input_file, "r", encoding="utf-8") as f:
        tasks = json.load(f)

    print("=" * 70)
    print("FORMAL VERIFICATION AUDIT")
    print("=" * 70)
    print(f"Tasks loaded: {len(tasks)}")
    print()

    audited_tasks = []

    for task in tasks:
        audited_tasks.append(
            audit_task(task)
        )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            audited_tasks,
            f,
            indent=4,
            ensure_ascii=False
        )

    return audited_tasks

In [11]:
# =====================================================================
# CELL 5D: AUDIT SUMMARY
# =====================================================================

def summarize_audit(audited_tasks):

    total = len(audited_tasks)

    target_failures = sum(
        not t["verification"]["target_compilation"]
        for t in audited_tasks
    )

    synthesis_failures = sum(
        t["verification"]["synthesis_compilation"] is False
        for t in audited_tasks
    )

    equivalence_passed = sum(
        t["verification"]["equivalence_passed"] is True
        for t in audited_tasks
    )

    option_tasks = sum(
        t["verification"]["options_audited"]
        for t in audited_tasks
    )

    option_sound = sum(
        t["verification"]["options_sound"] is True
        for t in audited_tasks
    )

    leaks = sum(
        t["verification"]["distractor_leak_detected"]
        for t in audited_tasks
    )

    fully_consistent = sum(
        t["verification"]["fully_consistent"]
        for t in audited_tasks
    )

    print("=" * 70)
    print("FORMAL VERIFICATION AUDIT SUMMARY")
    print("=" * 70)

    print(f"Tasks loaded:                 {total}")
    print(f"Target compilation failures:  {target_failures}")
    print(f"Synthesis compilation errors: {synthesis_failures}")
    print()

    if total:
        print(
            f"Exact equivalence:            "
            f"{equivalence_passed}/{total} "
            f"({equivalence_passed / total * 100:.1f}%)"
        )

    if option_tasks:
        print(
            f"Sound option sets:             "
            f"{option_sound}/{option_tasks} "
            f"({option_sound / option_tasks * 100:.1f}%)"
        )

    print(
        f"Distractor leaks:              {leaks}"
    )

    print(
        f"Fully consistent tasks:        "
        f"{fully_consistent}/{total} "
        f"({fully_consistent / total * 100:.1f}%)"
    )

    print("=" * 70)

    # --------------------------------------------------------------
    # Only print problematic tasks
    # --------------------------------------------------------------

    problems = []

    for task in audited_tasks:

        v = task["verification"]

        if not v["fully_consistent"]:

            problems.append(
                (
                    task.get("task_id"),
                    v
                )
            )

    if problems:

        print()
        print("⚠️ TASKS REQUIRING REVIEW")
        print("-" * 70)

        for task_id, v in problems:

            print(
                f"Task {task_id}: ",
                end=""
            )

            if v["counterexample"] is not None:
                print(
                    f"non-equivalent "
                    f"(counterexample='{v['counterexample']}')"
                )

            elif v["distractor_leak_detected"]:
                print("distractor leak")

            elif v["errors"]:
                print(v["errors"][0])

            elif v["options_sound"] is False:
                print("incorrect option labeling")

            else:
                print("failed consistency checks")

    else:

        print()
        print("✅ No consistency failures detected.")

In [ ]:
audited_tasks = run_audit(
    INPUT_FILE,
    OUTPUT_FILE
)

summarize_audit(audited_tasks)

## Cell 6: Distractor Error Classification Engine &amp; Plausibility Score
This cell implements the pedagogical quality evaluator based on the **Okuboyejo et al. (2025)** student error taxonomy, enhanced with Dr. Okuboyejo's approved **"Machine Noise / Gibberish"** bucket.
### Classification Architecture:
1. **Machine Noise / Gibberish Bucket:** Flags distractors that contain forbidden characters outside the problem's restricted alphabet (\\(\Sigma\\)), empty string anomalies, or unparsed junk.
2. **Slight Error Bucket:** Flags distractors with an edit distance (Levenshtein) of \\(\le 1\\) from a valid string accepted by the target DFA (minor slip: missing/added single character).
3. **Logical Error Bucket:** Flags valid strings over \\(\Sigma\\) that fail the target DFA due to structural misconceptions (e.g., prefix/suffix matches, swapped orderings, or partial repetition). - *Omitted Restriction:* Accepts over-generalized strings. - *Additional / Incorrect Restriction:* Rejects valid edge cases or enforces unintended constraints.
4. **Plausibility Score Calculation:** Computes the percentage of LLM-synthesized distractors that represent realistic student misconceptions versus synthetic AI noise: $$\text{Plausibility Score} = \frac{\text{Count of Taxonomy-Compliant Distractors}}{\text{Total Distractors Evaluated}} \times 100\%$$

###Cell 6A — clean utility functions

In [ ]:
# =====================================================================
# CELL 6A: DISTRACTOR CLASSIFICATION UTILITIES
# =====================================================================

import itertools


def levenshtein_distance(s1, s2):
    """
    Calculate the Levenshtein edit distance between two strings.
    """

    if len(s1) < len(s2):
        s1, s2 = s2, s1

    if len(s2) == 0:
        return len(s1)

    previous_row = list(range(len(s2) + 1))

    for i, c1 in enumerate(s1):

        current_row = [i + 1]

        for j, c2 in enumerate(s2):

            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = (
                previous_row[j] + (c1 != c2)
            )

            current_row.append(
                min(
                    insertions,
                    deletions,
                    substitutions
                )
            )

        previous_row = current_row

    return previous_row[-1]


def generate_sample_accepted_strings(
    dfa,
    alphabet,
    max_len=4,
    max_samples=100
):
    """
    Generate a bounded sample of strings accepted by a DFA.

    This is intentionally a sample, NOT an exhaustive representation
    of the language.
    """

    accepted = []

    alphabet = sorted(set(alphabet))

    for length in range(max_len + 1):

        for candidate_tuple in itertools.product(
            alphabet,
            repeat=length
        ):

            candidate = "".join(candidate_tuple)

            if dfa.accepts(candidate):

                accepted.append(candidate)

                if len(accepted) >= max_samples:
                    return accepted

    return accepted

###Cell 6B — classifier

In [ ]:
# =====================================================================
# CELL 6B: DISTRACTOR TAXONOMY CLASSIFIER
# =====================================================================

def classify_distractor(
    option_str,
    target_dfa,
    alphabet,
    sample_accepted_strings,
    max_length=25
):
    """
    Classify an intended distractor.

    Categories:
        - Machine Noise / Gibberish
        - Slight Error
        - Logical Error
        - Distractor Leak (Valid Answer)
    """

    alphabet = set(alphabet)

    # --------------------------------------------------------------
    # 1. Check for symbols outside Σ
    # --------------------------------------------------------------

    forbidden_symbols = [
        char
        for char in option_str
        if char not in alphabet
    ]

    if forbidden_symbols:

        return (
            "Machine Noise / Gibberish",
            "Contains symbol(s) outside the task alphabet: "
            + ", ".join(
                repr(x)
                for x in sorted(set(forbidden_symbols))
            )
        )

    # --------------------------------------------------------------
    # 2. Excessive length
    # --------------------------------------------------------------

    if len(option_str) > max_length:

        return (
            "Machine Noise / Gibberish",
            f"String exceeds configured maximum length "
            f"of {max_length}."
        )

    # --------------------------------------------------------------
    # 3. Distractor leak
    # --------------------------------------------------------------

    if target_dfa.accepts(option_str):

        return (
            "Distractor Leak (Valid Answer)",
            "String is accepted by the target DFA."
        )

    # --------------------------------------------------------------
    # 4. Surface-level slight error
    # --------------------------------------------------------------

    closest_string = None
    closest_distance = None

    for valid_str in sample_accepted_strings:

        distance = levenshtein_distance(
            option_str,
            valid_str
        )

        if closest_distance is None or distance < closest_distance:

            closest_distance = distance
            closest_string = valid_str

    if closest_distance is not None and closest_distance <= 1:

        return (
            "Slight Error",
            f"Levenshtein distance {closest_distance} "
            f"from sampled valid string '{closest_string}'."
        )

    # --------------------------------------------------------------
    # 5. Structurally invalid but alphabet-valid
    # --------------------------------------------------------------

    return (
        "Logical Error",
        "Uses only symbols from Σ but is rejected "
        "by the target DFA."
    )

###Cell 6C — actual audit

In [ ]:
# =====================================================================
# CELL 6C: DATASET DISTRACTOR CLASSIFICATION
# =====================================================================

import json
import os


INPUT_FILE = "audited_tasks_llama.json"
OUTPUT_FILE = "classified_tasks_llama.json"


def classify_dataset(
    input_file,
    output_file,
    max_sample_length=4,
    max_accepted_samples=100,
    max_option_length=25
):

    if not os.path.exists(input_file):

        raise FileNotFoundError(
            f"Input file '{input_file}' not found. "
            "Run Cell 5 first."
        )

    with open(
        input_file,
        "r",
        encoding="utf-8"
    ) as f:

        audited_tasks = json.load(f)

    print(
        f"🔍 Classifying distractors across "
        f"{len(audited_tasks)} tasks..."
    )

    bucket_counts = {
        "Machine Noise / Gibberish": 0,
        "Slight Error": 0,
        "Logical Error": 0,
        "Distractor Leak (Valid Answer)": 0
    }

    total_options = 0
    total_intended_distractors = 0
    total_correct_options = 0
    total_leaks = 0

    classified_tasks = []

    for task in audited_tasks:

        task_copy = task.copy()

        task_id = task.get("task_id")
        target_regex = task.get("target_regex")
        alphabet = task.get("alphabet", [])
        options = task.get("options", [])

        # ----------------------------------------------------------
        # Compile target DFA
        # ----------------------------------------------------------

        try:

            target_dfa = compile_regex_to_dfa(
                target_regex,
                alphabet
            )

        except Exception as e:

            task_copy["classification_error"] = str(e)

            classified_tasks.append(task_copy)

            continue

        # ----------------------------------------------------------
        # Generate bounded valid examples
        # ----------------------------------------------------------

        sample_accepted = (
            generate_sample_accepted_strings(
                target_dfa,
                alphabet,
                max_len=max_sample_length,
                max_samples=max_accepted_samples
            )
        )

        classified_options = []

        for option in options:

            total_options += 1

            option_copy = option.copy()

            option_str = (
                option.get("string", "")
                .strip()
            )

            labeled_correct = bool(
                option.get(
                    "labeled_correct",
                    False
                )
            )

            # ------------------------------------------------------
            # Recalculate ground truth independently
            # ------------------------------------------------------

            actual_match = target_dfa.accepts(
                option_str
            )

            option_copy["actual_target_match"] = (
                actual_match
            )

            # ------------------------------------------------------
            # Intended correct answer
            # ------------------------------------------------------

            if labeled_correct:

                total_correct_options += 1

                if actual_match:

                    option_copy["taxonomy_category"] = (
                        "Correct Answer"
                    )

                    option_copy["taxonomy_rationale"] = (
                        "Labeled correct and accepted "
                        "by the target DFA."
                    )

                else:

                    option_copy["taxonomy_category"] = (
                        "Incorrectly Labeled Correct"
                    )

                    option_copy["taxonomy_rationale"] = (
                        "Labeled correct but rejected "
                        "by the target DFA."
                    )

            # ------------------------------------------------------
            # Intended distractor
            # ------------------------------------------------------

            else:

                total_intended_distractors += 1

                category, rationale = classify_distractor(
                    option_str,
                    target_dfa,
                    alphabet,
                    sample_accepted,
                    max_length=max_option_length
                )

                option_copy["taxonomy_category"] = category
                option_copy["taxonomy_rationale"] = rationale

                if category in bucket_counts:

                    bucket_counts[category] += 1

                if category == (
                    "Distractor Leak (Valid Answer)"
                ):

                    total_leaks += 1

            classified_options.append(
                option_copy
            )

        task_copy["options"] = classified_options

        classified_tasks.append(
            task_copy
        )

    # --------------------------------------------------------------
    # Save results
    # --------------------------------------------------------------

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            classified_tasks,
            f,
            indent=4,
            ensure_ascii=False
        )

    return {
        "tasks": classified_tasks,
        "bucket_counts": bucket_counts,
        "total_options": total_options,
        "total_intended_distractors": total_intended_distractors,
        "total_correct_options": total_correct_options,
        "total_leaks": total_leaks
    }

In [ ]:
# =====================================================================
# CELL 6D: DISTRACTOR QUALITY SUMMARY
# =====================================================================

results = classify_dataset(
    INPUT_FILE,
    OUTPUT_FILE
)

bucket_counts = results["bucket_counts"]

total_distractors = (
    results["total_intended_distractors"]
)

total_leaks = results["total_leaks"]

non_leak_distractors = (
    total_distractors - total_leaks
)

plausible_distractors = (
    bucket_counts["Slight Error"]
    + bucket_counts["Logical Error"]
)

print()
print("=" * 70)
print("PEDAGOGICAL QUALITY & DISTRACTOR TAXONOMY")
print("=" * 70)

print(
    f"Total options evaluated:       "
    f"{results['total_options']}"
)

print(
    f"Intended distractors:          "
    f"{total_distractors}"
)

print(
    f"Intended correct answers:      "
    f"{results['total_correct_options']}"
)

print()
print("DISTRACTOR BREAKDOWN")
print("-" * 70)

if total_distractors:

    print(
        f"Logical Errors:               "
        f"{bucket_counts['Logical Error']:3d} "
        f"({bucket_counts['Logical Error'] / total_distractors * 100:.1f}%)"
    )

    print(
        f"Slight Errors:                "
        f"{bucket_counts['Slight Error']:3d} "
        f"({bucket_counts['Slight Error'] / total_distractors * 100:.1f}%)"
    )

    print(
        f"Machine Noise / Gibberish:    "
        f"{bucket_counts['Machine Noise / Gibberish']:3d} "
        f"({bucket_counts['Machine Noise / Gibberish'] / total_distractors * 100:.1f}%)"
    )

    print(
        f"Distractor Leaks:             "
        f"{bucket_counts['Distractor Leak (Valid Answer)']:3d} "
        f"({bucket_counts['Distractor Leak (Valid Answer)'] / total_distractors * 100:.1f}%)"
    )

print()
print("QUALITY METRICS")
print("-" * 70)

if total_distractors:

    leak_free_rate = (
        non_leak_distractors
        / total_distractors
        * 100
    )

    print(
        f"Leak-Free Distractor Rate:     "
        f"{leak_free_rate:.1f}%"
    )

if non_leak_distractors:

    plausibility_score = (
        plausible_distractors
        / non_leak_distractors
        * 100
    )

    print(
        f"Non-Leak Plausibility Rate:    "
        f"{plausibility_score:.1f}%"
    )

print("=" * 70)

print(
    f"💾 Saved classifications to "
    f"'{OUTPUT_FILE}'"
)